# Control policy optimization

In this example, a symbolic policy is evolved for the pendulum swingup task. Gymnax is used for simulation of the pendulum environment, showing that Kozax can easily be extended to external libraries.

In [ ]:
!pip install mujoco==3.9.0
!pip install mujoco_mjx==3.9.0
!pip install brax
!pip install playground
!pip install kozax
!pip install gymnax
!pip install jaxtyping
!pip install playground warp-lang==1.12.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: mujoco
    Found existing installation: mujoco 3.10.0
    Uninstalling mujoco-3.10.0:
      Successfully uninstalled mujoco-3.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mujoco-mjx 3.10.0 requires mujoco>=3.10.0.dev0, but you have mujoco 3.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 81.1 MB/s eta 0:00:00
  Attempting uninstall: mujoco_mjx
    Found existing installation: mujoco-mjx 3.10.0
    Uninstalling mujoco-mjx-3.10.0:
      Successfully uninstalled mujoco-mjx-3.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 7.3 MB/s eta 0:00:00
  Attempting uninstall: warp-lang
    Found existing installation: warp-lang 1.14.0
    

In [ ]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Setting environment variable to use GPU rendering:
env: MUJOCO_GL=egl
Checking that the installation succeeded:
Installation successful.


In [ ]:
# @title Import packages for plotting and creating graphics
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
print("Installing mediapy:")
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

Installing mediapy:


In [ ]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
import jax.numpy as jnp
import jax.random as jr
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

In [ ]:
from mujoco_playground import registry

class MujocoGymnaxWrapper:

    def __init__(self, env_name = None, env_instance = None, config_overrides = None):
        if env_instance is not None:
            self.env = env_instance
        else:
            self.env = registry.load(env_name)
        # self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, key, state, action, params=None):

      next_state = self.env.step(state, action)

      done = next_state.done
      obs = next_state.obs
      reward = next_state.reward

      return obs, next_state, reward, done, {}

    def render(self, rollout, camera = None):
      if camera == None:
        return self.env.render(rollout)
      else:
        return self.env.render(rollout, camera=camera)

    @property
    def dt(self):
        return self.env.dt

In [ ]:
def compute_trajectory_py(key, env, policy, T=1000, stride=1):
  jit_reset = jax.jit(env.reset)
  jit_step = jax.jit(env.step)
  obs, env_state = jit_reset(key)
  states = []

  for t in range(T):
      action = policy(obs)
      obs, env_state, reward, done, _ = jit_step(key, env_state, action)

      if t % stride == 0:
          states.append(env_state)

      if done:
          break

  return states

In [ ]:
def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)

        def step_fn(carry, t):
            obs, env_state, done, key = carry
            key, subkey = jr.split(key)
            action = policy(obs)
            obs, env_state, reward, done_new, _ = env.step(subkey, env_state, action, None)
            done = done | done_new.astype(bool)
            reward = jnp.where(done, 0.0, reward)
            return (obs, env_state, done, key), (obs, reward, action, done)

        (_, _, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn,
            (obs, env_state, False, key),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

## Cartpole

Action Space: [-1, 1], where action represents the numerical force applied to the cart

Observation Space: $ℝ^{4}$

0.   car_position
1.   vertical_angle_pole
2.   linear_velocity_cart
3.   angular_velocity_pole

Reward: +1 is awarded for each timestep that the pole is upright.

The episode terminates when episode duration reaches 1000 timestep or the absolute value of the vertical angle between the pole and the cart is greater than 0.2 radians.

### Visualize best solution

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
T = 1000

Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.


In [ ]:
import mujoco
import warp as wp

print("mujoco:", mujoco.__version__)
print("warp:", wp.__version__)

mujoco: 3.9.0
warp: 1.12.1


## Kozax

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.2*obs[2] + obs[3]*(obs[2] + obs[3]*obs[4]) + obs[3] + 1.6*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

Module mujoco.mjx.warp.ffi 98d2031 load on device 'cuda:0' took 535.06 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.smooth 95a0b8b load on device 'cuda:0' took 7462.88 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 981fdaf load on device 'cuda:0' took 3390.84 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward f60f76d load on device 'cuda:0' took 2405.14 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.passive 1a5e55c load on device 'cuda:0' took 1650.98 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.forward 8ebfcb1 load on device 'cuda:0' took 1980.37 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.support dd3f924 load on device 'cuda:0' took 340.78 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_2f3988cb 280e0ec load on device 'cuda:0' took 5267.56 ms  (compiled)
Module mujoco.mjx.third_party.mujoco_warp._src.solver b116efb load on device 'cuda

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + obs[2]*(2*obs[1] + 4.22) + obs[3] + 1.63*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[999.687 999.897 999.883 999.441 999.824 999.706 999.807 999.6   999.872 999.55 ]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([obs[0] + 6.13*obs[2] + obs[3] + 1.62*obs[4]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[999.686 999.897 999.883 999.441 999.824 999.706 999.807 999.6   999.872 999.55 ]


## Modi

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([
    1.44*obs[0] + 5.87*obs[2] + 1.47*obs[3] + 1.47*obs[4]
    + jnp.sin(obs[0]*obs[3] + 2*obs[2] + obs[4])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[999.681 999.895 999.882 999.439 999.824 999.703 999.805 999.592 999.871 999.542]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([
    obs[0] + 6.18*obs[2] + obs[3] + 1.65*obs[4]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[999.686 999.897 999.883 999.441 999.824 999.706 999.807 999.6   999.872 999.55 ]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CartpoleBalance')
policy = lambda obs: jnp.array([
    obs[0] + 6.18*obs[2] + obs[3] + 1.65*obs[4]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[999.686 999.897 999.883 999.441 999.824 999.706 999.807 999.6   999.872 999.55 ]


## Reacher

Action space: $[-1, 1]^2$, where the first action is the torque applied at the first hinge (connecting the link to the point of fixture), and the second action the torque applied at the second hinge (connecting the two links)

Observation space: $ℝ^{11}$

0.   cos(joint0)
1.   cos(joint1)
2.   sin(joint0)
3.   sin(joint1)
4.   target_x
5.   target_y
6.   ang_vel joint 0
7.   ang_vel joint 1
8.   diff x-value
9.   diff y-value
10.  (diff z-value)

The reward consists of two parts:

  - *reward_dist*: distance between *fingertip* of the reacher and the target,  with a more negative value assigned when *fingertip* is further away.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as the negative squared Euclidean norm of the action.


   The episode terminates when episode duration reaches a 1000 timesteps


### Visualize best solution

In [ ]:
env = MujocoGymnaxWrapper("ReacherEasy")
T = 1000
#env = EpisodeWrapper(envs.get_environment("reacher"), episode_length=T, action_repeat=1)

## Kozax

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    -0.591*obs[0]*obs[2] + obs[2]**2 + 4*obs[2] + 7*obs[3],
    -7*obs[2] + 0.321*obs[4]*(obs[0]*(2*obs[2]*obs[5] - 0.881) - 0.844)
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[560. 958. 935. 848. 876.  10. 120. 967. 906. 954.]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    -obs[2] + 15*obs[3],
    (-2.81*obs[0]*obs[3] - 5*obs[1])*(2*obs[0]*obs[3] + obs[1]*obs[3] + obs[2] + obs[3])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[  0. 963.  21.   0.   8. 360. 978. 949. 972. 966.]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    obs[0]*obs[2] + 2*obs[0]*obs[3] + 10*obs[2],
    jnp.sin(
        obs[3]*(
            1.34*obs[1]
            + 1.34*obs[3]*(
                1.34*obs[1]
                + 1.34*obs[3]
                + 1.34*(obs[4]**2 + obs[4])*(obs[0] + obs[1] - 0.446)
            )
        )
    )
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[962.   0. 938. 896. 951. 875.  43. 964.  32.  22.]


## Modi

In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    obs[1]*obs[2] + 2*obs[2]*obs[3] + 4*obs[2]
    + 2*obs[3]*(obs[0] + (obs[2] + obs[3])*(obs[0] + obs[1]*obs[2] + obs[5]))
    + 4*obs[3],

    4*obs[2]
    + 4*obs[3]*(obs[0] + (obs[2] + obs[3])*(obs[0] + obs[1]*obs[2] + obs[5]))
    + (obs[2] + obs[3])*(obs[0] + obs[1]*obs[2] + obs[5])
    + (3*obs[2] + 2*obs[3]*(obs[0] + (obs[2] + obs[3])*(obs[0] + obs[1]*obs[2] + obs[5])))
      *(obs[0] + obs[2]*obs[3] + obs[3])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[874. 974. 959. 942. 837. 821. 718. 951. 980.   6.]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    obs[0]*obs[3]*(obs[0]*obs[2] + jnp.sin(obs[2]*(-obs[4] + 0.225*jnp.sin(obs[2]*(-obs[1] + 2*obs[3])) - 0.225)) + 2.35)
    + obs[2]*(-obs[1] + 2*obs[3])
    + obs[2]*(-obs[4] + 0.225*jnp.sin(obs[2]*(-obs[1] + 2*obs[3])) - 0.225)
    + obs[2]
    + 2*obs[3]
    + 0.225*jnp.sin(obs[2]*(-obs[1] + 2*obs[3])),

    obs[0]*obs[2]
    + obs[0]*obs[3]*(obs[0]*obs[2] + jnp.sin(obs[2]*(-obs[4] + 0.225*jnp.sin(obs[2]*(-obs[1] + 2*obs[3])) - 0.225)) + 2.35)
    + jnp.sin(obs[2]*(-obs[1] + 2*obs[3]))
    + jnp.sin(obs[2]*(-obs[4] + 0.225*jnp.sin(obs[2]*(-obs[1] + 2*obs[3])) - 0.225))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[986.  45. 956. 944.   0. 628.  74. 887.  77.  19.]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('ReacherEasy')
policy = lambda obs: jnp.array([
    3*obs[0]*jnp.sin(jnp.sin(obs[3] + (obs[3] + jnp.sin(obs[3]))*jnp.sin(2*obs[3])))
    + 8*obs[2]
    + 7*obs[3]
    + obs[4]*(obs[2] + obs[3])**2
    + (obs[3] + jnp.sin(obs[3]))*jnp.sin(2*obs[3])
    + jnp.sin(obs[3]),

    obs[0]*jnp.sin(jnp.sin(obs[3] + (obs[3] + jnp.sin(obs[3]))*jnp.sin(2*obs[3])))
    + obs[3]
    + (obs[2] + obs[3])**2
    + jnp.sin(2*obs[3])
    + jnp.sin(obs[3] + (obs[3] + jnp.sin(obs[3]))*jnp.sin(2*obs[3]))
    + jnp.sin(jnp.sin(obs[3] + (obs[3] + jnp.sin(obs[3]))*jnp.sin(2*obs[3])))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[913. 613. 911. 901. 940. 915. 853. 981. 835.   0.]


## Swimmer

Action space: $[-1, 1]^2$, where the first action is the torque applied on the first rotor, and the second action the torque applied on the second rotor.

Observation space: $ℝ^{8}$

0.   angle_front_tip
1.   angle_second_rotor
2.   angle_third_rotor
3.   velocity_tip_x-axis
4.   velocity_tip_y-axis
5.   angular_velocity_front_tip
6.   angular_velocity_second_rotor
7.   angular_velocity_third_rotor

The reward consists of two parts:

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps

### Visualize best solution

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
T = 1000

## Kozax

In [ ]:
from mujoco_playground._src.dm_control_suite.swimmer import Swim

Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    5*obs[10] + obs[12] - 2.12*obs[2] + obs[5] + 1.77*obs[9],
    obs[2] + obs[9] + (-obs[3] + obs[4])*(obs[0] + obs[9] + (obs[0] + obs[2])*(-obs[2] - 2*obs[3] + 2*obs[6]*(obs[2] + obs[6] + obs[9])))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

Module mujoco.mjx.third_party.mujoco_warp._src.sensor 7a62d05 load on device 'cuda:0' took 7811.98 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f9126636 4908da2 load on device 'cuda:0' took 4843.53 ms  (compiled)
Module solve_init_jaref__locals__kernel_6e8468b3 6e8468b load on device 'cuda:0' took 150.80 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_f619f9d8 f619f9d load on device 'cuda:0' took 214.84 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_84cf3b0a 84cf3b0 load on device 'cuda:0' took 145.18 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_dc766f91 aa19e9d load on device 'cuda:0' took 531.44 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_8204ed89 a39c2f0 load on device 'cuda:0' took 3969.63 ms  (compiled)
Module linesearch_iterative__locals__kernel_94acdc55 9c914fd load on device 'cuda:0' took 1056.41 ms  (compiled)
[267.634 255.458 711.224  27.075  14.01  428.743  23.

In [ ]:
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    obs[11]*(2*obs[0] + 2*obs[1] + obs[12] + 4*obs[2] + 3*obs[6]),
    (-obs[0]*(2*obs[3] + jnp.sin(obs[3]) + jnp.sin(obs[3] + 2*jnp.sin(obs[3]))) + obs[0] + 2*obs[2] - obs[9] - jnp.sin(obs[3]))*jnp.cos(obs[6])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 12.536  16.121 840.797  27.684  13.835  37.865   6.353  10.287  13.56  341.262]


In [ ]:
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    obs[9] + (obs[6] - obs[9] + (obs[6] - 0.957)*(-obs[0] + obs[6] - 0.957)*(-obs[1] + obs[2] + obs[6] - 0.957) - 0.957)*jnp.sin(obs[2] + obs[8]),
    4*obs[0] - 6*obs[1] - obs[10] + obs[2] + obs[7] - obs[8] + 1.01
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[280.763  11.941 894.36   36.79  377.348  23.267  29.34  198.311  86.544 606.461]


## Modi

In [ ]:
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    obs[0]
    + obs[12]*(2*obs[8] - 0.0846)*(obs[2] + obs[7] + obs[9])*(obs[0] + obs[2] + obs[4] + obs[7])
    + 2*obs[7]
    + 2*obs[9],

    obs[0]
    + 2*obs[12]*(2*obs[8] - 0.0846)*(obs[2] + obs[7] + obs[9])*(obs[0] + obs[2] + obs[4] + obs[7])
    + 3*obs[2]
    + obs[3]*(obs[2] + obs[3]*(obs[12]*(2*obs[8] - 0.0846)*(obs[2] + obs[7] + obs[9])*(obs[0] + obs[2] + obs[4] + obs[7]) + obs[2] + obs[9]))
    + 2*obs[3]*(obs[12]*(2*obs[8] - 0.0846)*(obs[2] + obs[7] + obs[9])*(obs[0] + obs[2] + obs[4] + obs[7]) + obs[2] + obs[9])
    + obs[4]
    + 2*obs[7]
    + 2*obs[9]
    + (2*obs[8] - 0.0846)*(obs[2] + obs[7] + obs[9])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[177.766  11.786 900.767  35.742  14.086  21.9    11.597   9.76   60.66  643.165]


In [ ]:
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    2*jnp.sin(obs[1]*obs[3]),

    obs[0]
    + 2*obs[1]*obs[3]
    + obs[12]
    + obs[2]
    + 2*obs[3]*(-obs[0] + obs[1] - obs[5])
    + obs[5]
    + (obs[2] + obs[3]*(-obs[0] + obs[1] - obs[5]) - obs[3])
      *(-obs[3] + obs[5] + 2*jnp.sin(obs[1]*obs[3]))
    + 4*jnp.sin(obs[1]*obs[3])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 32.995  11.623 890.912  30.058 118.027  22.494  23.144  10.377 103.198 576.589]


In [ ]:
Swimmer_env = Swim(n_links=3)
mujoco_env = MujocoGymnaxWrapper(env_instance = Swimmer_env)
policy = lambda obs: jnp.array([
    obs[0] - obs[1] - obs[10]
    + 4.23*obs[11]*obs[2]
    + 1.58*obs[12]*obs[2]*obs[9]
    - 3.15*obs[2]
    + obs[4]
    + 2*obs[5]*obs[9]
    - obs[6]
    + obs[9],

    2*obs[0]
    - 4*obs[1]
    - obs[10]
    + 16.9*obs[11]*obs[2]
    - 1.58*obs[12]*obs[2]*obs[9]
    + 4*obs[4]
    + 2*obs[5]*obs[9]
    - 4*obs[6]
]
)
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[297.094  12.297 616.514 150.59  383.667  24.366  34.33  228.602 164.441 667.77 ]


## Hopper

Action space: $[-1, 1]^3$, where the first action is the torque applied on the thigh rotor, and the second action the torque applied on the leg rotor, and the thrid action the torque applied to the foot rotor.

Observation space: $ℝ^{11}$

0.   z-coordinate of the top
1.   angle_top
2.   angle_thigh_joint
3.   angle_leg_joint
4.   angle_foot_joint
5.   velocity_x-coordinate_top
6.   velocity_z-coordinate_top
7.   angular_velocity_angle_top
8.   angular_velocity_thigh_hinge
9.   angular_velocity_leg_hinge
10.  angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*: +1 reward for every timestep that the hopper is alive.

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps, the height of the hopper becomes less than 0.7 metres, or the absolute value of the angle of the thigh joint is less than 0.2 radians

### Visualize

In [ ]:
env = MujocoGymnaxWrapper("HopperHop")
T = 1000

## Kozax

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    obs[10] + obs[13]**2*obs[4]*(obs[10] + obs[13]) + obs[7] + obs[8],
    3*obs[10] + obs[12] + obs[6] + 1.34,
    -obs[0] - obs[10] - 3*obs[6] + jnp.cos(obs[0] + obs[1]),
    2*obs[10] - 1.29
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver 88f7770 load on device 'cuda:0' took 146.82 ms  (compiled)
Module _nxn_broadphase__locals__kernel_d2d200c6 d2d200c load on device 'cuda:0' took 594.62 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_ca3f8f98 ca3f8f9 load on device 'cuda:0' took 1006.68 ms  (compiled)
Module _efc_contact_init__locals__kernel_703a08a4 703a08a load on device 'cuda:0' took 179.57 ms  (compiled)
Module _efc_contact_jac_dense__locals__kernel_11a6f18c 927acba load on device 'cuda:0' took 562.56 ms  (compiled)
Module _efc_contact_update__locals__kernel_c77ca81d c77ca81 load on device 'cuda:0' took 298.61 ms  (compiled)
Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_f956cf88 226c0f7 load on device 'cuda:0' took 4041.35 ms  (compiled)
Module solve_init_jaref__locals__kernel_173cbb76 173cbb7 load on device 'cuda:0' took 148.25 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_d21c2e6d d21c2e6

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    obs[10]*obs[4],
    obs[10]*obs[7] + obs[4]*obs[7] - obs[4] + 2*obs[7] - obs[9],
    -obs[12] + 5*obs[3] - 0.114,
    obs[10] + jnp.sin(obs[14]) + jnp.sin(obs[3] + jnp.sin(obs[9]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 50.062  55.835 103.078  41.025  59.774  46.909   0.     83.589  86.485  72.984]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    jnp.sin(jnp.sin(jnp.sin(jnp.sin(jnp.sin(obs[12]))))),
    obs[10] + 3*obs[4] + 4*obs[6],
    obs[0] + obs[11]*obs[3]*obs[6] + obs[7] + obs[8],
    jnp.sin(jnp.cos(obs[12])) + jnp.sin(jnp.cos(jnp.sin(jnp.cos(obs[12])) + jnp.sin(jnp.cos(jnp.sin(jnp.cos(obs[12]))))))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 0.     0.    49.454 48.054  0.071  0.     0.    55.669  0.    49.893]


## Modi

In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    0,

    -2*obs[0]*obs[11]
    + 3*obs[0]*obs[3]
    - obs[0]*(obs[0] + obs[10] + obs[4])
    + obs[0]
    + obs[10]
    - 2*obs[11]*(obs[14] + obs[4] + obs[7])
    + 4*obs[3]
    + 4*obs[4]
    + 3*obs[8],

    obs[0]*obs[3] + obs[8],

    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 0.     0.    29.344 32.354  0.     0.    32.865 26.848  0.    20.944]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    2*obs[0] + obs[7],

    obs[10] - obs[11]
    + jnp.cos(obs[1])
    + jnp.cos(-2*obs[10] + obs[11] + jnp.cos(obs[1])),

    2*obs[0]
    + obs[11]
    + 2*obs[3]
    + 2*obs[5]
    + (2*obs[0] + obs[7])
      *(-2*obs[0] + obs[3] + obs[5] + jnp.cos(-2*obs[10] + obs[11] + jnp.cos(obs[1])))
    + jnp.cos(-2*obs[10] + obs[11] + jnp.cos(obs[1])),

    obs[10] - jnp.cos(obs[1])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[51.364 26.723 37.244 53.643 20.863 39.085 34.496 41.529 36.987 49.234]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('HopperHop')
policy = lambda obs: jnp.array([
    0,

    obs[10]
    - obs[11]
    + obs[12]
    + obs[13]*(obs[2]*obs[4] + obs[6]**2)
    + obs[2]**2
    + obs[2]*obs[4]*(-obs[5] + obs[8])
    + 2*obs[6],

    obs[2]*obs[4],

    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[ 0.     0.    36.191 35.188  0.088  0.     0.    33.703  0.    37.048]


## Walker2d

Action space: $[-1, 1]^6$, where the first action is the torque applied on the thigh rotor, the second action the torque applied on the leg rotor, the third action the torque applied to the foot rotor, the fourth action is the torque applied on the left thigh rotor, the fifth action the torque applied on the left leg rotor, and the sixth action the torque applied to the left foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_top
1. angle_top
2. angle_thigh_joint
3. angle_leg_joint
4. angle_foot_joint
5. angle_left_thigh_joint
6. angle_left_leg_joing
7. angle_left_foot_joint
8. velocity_x-coordinate_top
9. velocity_z-coordinate_top
10. angular_velocity_top
11. angular_velocity_thigh_hinge
12. angular_velocity_leg_hinge
13. angular_velocity_foot_hinge
14. angular_velocity_thigh_hinge
15. angular_velocity_leg_hinge
16. angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*:
    +1 for each timepoint that the walker is alive
  - *reward_forward*:
    Reward of walking forward, measured as *(x-coordinate before action - x-coordinate after action) / dt*.
  - *reward_ctrl*:
    Negative reward for penalising the walker if it takes too large actions, measured as *-coefficient **x** sum(action<sup>2</sup>)*.

   The episode terminates when episode duration reaches a 1000 timesteps, when the height of the walker is not in the range [0.7, 2], and the absolute value of the angle is not in the range [-1, 1].

### Visualize best solution

In [ ]:
env = MujocoGymnaxWrapper("WalkerWalk")
T = 1000

## Kozax

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    obs[18] + jnp.sin(jnp.sin(obs[18])),
    obs[10] + obs[14] - obs[16],
    obs[1] + jnp.cos(jnp.sin(obs[12] - obs[6])),
    obs[1] + 2*obs[18] - 1.07,
    obs[0] + obs[11] - obs[7] - 0.827,
    jnp.sin(obs[11] + obs[4]) + 0.624
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

Module _tile_cholesky_factorize_solve__locals__cholesky_factorize_solve_cdb981f2 c75d2e6 load on device 'cuda:0' took 4166.31 ms  (compiled)
Module solve_init_jaref__locals__kernel_051c9471 051c947 load on device 'cuda:0' took 130.32 ms  (compiled)
Module mul_m_dense__locals___mul_m_dense_86090c8a 86090c8 load on device 'cuda:0' took 262.93 ms  (compiled)
Module update_constraint_gauss_cost__locals__kernel_0aec6e48 0aec6e4 load on device 'cuda:0' took 148.09 ms  (compiled)
Module update_gradient_JTDAJ_dense_tiled__locals__kernel_2b23959a 84da6db load on device 'cuda:0' took 751.07 ms  (compiled)
Module update_gradient_cholesky__locals__kernel_97d1b30b 6b32db2 load on device 'cuda:0' took 4493.02 ms  (compiled)
Module linesearch_iterative__locals__kernel_6962ca73 89f17c9 load on device 'cuda:0' took 989.14 ms  (compiled)
[255.816 324.743 326.308 256.602 255.319 292.164 257.535 290.454 314.176 199.473]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    3*obs[0] + obs[21],
    obs[7]*(obs[14]*obs[7] - obs[7]),
    4*obs[17]**2,
    obs[13] + 3*obs[21],
    obs[0] + obs[12]*obs[8] + obs[9],
    obs[10] - obs[3]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[286.654 281.65  282.569 234.364 308.975 291.702 285.649 290.315 224.39  194.977]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    3*obs[18],
    -obs[1]*obs[3] + obs[14],
    jnp.cos(obs[10]) + jnp.cos(jnp.cos(jnp.cos(obs[4]))),
    4*obs[18],
    3*obs[14] + obs[9],
    obs[14]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[272.288 263.947 212.877 286.494 294.424 184.447 295.429 274.594 235.297 299.392]


## Modi

In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    2*obs[1]*(obs[10] + obs[12]*obs[8])
    + 4*obs[14]*(obs[11] + obs[14]*(obs[1] + obs[14] + obs[8]) - obs[21])
    + 4*obs[21]
    + obs[22]
    + 0.33,

    2*obs[1]
    + obs[10]
    + obs[12]*obs[8]
    + 4*obs[14]
    + obs[8],

    obs[14]*(obs[1] + obs[14] + obs[8]),

    2*obs[21],

    obs[1]*(obs[10] + obs[12]*obs[8]) + obs[12]*obs[8],

    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[316.619 345.502 280.08  368.348 341.867 390.875 363.044 344.176 341.779 329.082]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    obs[18] - obs[2] + obs[21] + jnp.cos(obs[11]) + 0.127,

    -obs[18] + obs[4] - jnp.cos(jnp.cos(obs[21])),

    jnp.cos(obs[21]) + jnp.cos(obs[7]),

    3*obs[18] - obs[2] + 2*obs[21] + jnp.cos(jnp.cos(obs[21])) + 0.127,

    obs[11]*obs[9]
    *(-obs[18] - obs[21] - jnp.cos(jnp.cos(obs[21])))
    *(obs[18] - obs[2] + obs[21] + jnp.cos(obs[11]) + jnp.cos(obs[7]) + 0.127),

    jnp.cos(obs[11]) + jnp.cos(jnp.cos(obs[21]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[200.299 204.469 314.776 235.352 201.337 276.866 266.976 266.101 300.673 138.327]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('WalkerWalk')
policy = lambda obs: jnp.array([
    -obs[14] - 2*obs[17] + 4*obs[21] + obs[8] - 1.84,

    obs[4]*obs[7],

    obs[10]*obs[3]*jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
    - obs[13]
    + obs[16]
    + obs[3]
    + (obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21])
    + (obs[8] - 1.84)
      *(obs[10]*obs[3]*jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
        - obs[13] + obs[16] + obs[3]),

    obs[10]*obs[3]*jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
    + obs[13]
    + obs[16]
    + 3*obs[21]
    + jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
    + jnp.sin((obs[8] - 1.84)
        *(obs[10]*obs[3]*jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
          - obs[13] + obs[16] + obs[3])),

    obs[4]*obs[7]
    + jnp.sin((obs[8] - 1.84)
        *(obs[10]*obs[3]*jnp.sin((obs[13] + obs[21])*(-obs[14] - obs[17] + 2*obs[21]))
          - obs[13] + obs[16] + obs[3])),

    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[370.719 262.261 309.53  367.203 326.679 333.46  331.307 209.457 285.203 294.864]


## Half cheetah

Action space: $[-1, 1]^6$, where the first action is the torque applied on the back thigh rotor, the second action the torque applied on the back shin rotor, the third action the torque applied to the back foot rotor, the fourth action is the torque applied on the front thigh rotor, the fifth action the torque applied on the front shin rotor, and the sixth action the torque applied to the front foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_mass
1. w-orientation_front_tip
2. y-orientation_front_tip
3. angle_back_thigh_rotor
4. angle_back_shin_rotor
5. angle_back_foot_rotor
6. velocity_tip_along_y-axis
7. angular_velocity_front_tip
8. angular_velocity_second_rotor
9. x-coordinate_front_tip
10. y-coordinate_front_tip
11. angle_front_tip
12. angle_second_rotor
13. angle_second_rotor
14. velocity_tip_along_x-axis
15. velocity_tip_along_y-axis
16. angular_velocity_front_tip
17. angular_velocity_second_rotor

The reward consists of two parts:

  - *reward_run*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps.

### Visualize best solution

In [ ]:
env = MujocoGymnaxWrapper("CheetahRun")
# env = EpisodeWrapper(envs.get_environment(env_name="halfcheetah"), episode_length=1000, action_repeat=1)
T = 1000

## Kozax

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    obs[1] - obs[15],
    obs[5],
    obs[1] + obs[11],
    obs[2],
    obs[12],
    1.75
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[274.479 278.729 264.012 296.48  294.525 244.15  135.257 243.986 258.053 266.829]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    3*obs[11],
    jnp.cos(jnp.cos(jnp.cos(obs[2]*obs[5]))),
    obs[1] + obs[10] + obs[11],
    2*obs[0] + obs[1]*obs[6],
    obs[0] - obs[11],
    obs[4]*(obs[2] + 0.256)
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[135.5    76.68  257.092 275.702 282.553 246.632 261.532 164.505 312.732 328.403]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    obs[1] + obs[11],
    jnp.cos(2*obs[0] + jnp.cos(obs[0])),
    obs[11]*obs[3]**2 - obs[3],
    2*obs[0] + obs[1]*obs[6],
    obs[10] - obs[11] + obs[2] + 0.807,
    obs[4]**2
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[320.745 379.107 358.162 385.786 336.995 334.997 329.397 316.839 341.817 365.036]


## Modi

In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    5*obs[0]
    - obs[10]
    + obs[11]
    + 2*jnp.sin(obs[9])
    - jnp.sin(obs[11] + jnp.sin(obs[9]))
    - jnp.cos(obs[8]*jnp.sin(jnp.sin(jnp.sin(jnp.sin(obs[15])))))
    + jnp.cos(obs[0] + obs[2]),

    0,

    3*obs[0]
    - obs[10]
    - jnp.sin(obs[11] + jnp.sin(obs[9]))
    - jnp.cos(obs[8]*jnp.sin(jnp.sin(jnp.sin(jnp.sin(obs[15]))))),

    0,
    0,

    obs[0]
    + obs[2]
    + jnp.cos(obs[8]*jnp.sin(jnp.sin(jnp.sin(jnp.sin(obs[15])))))
    + jnp.cos(obs[0] + obs[2])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[248.874 285.53  258.969 263.085 276.086 254.769 271.678 241.969 269.057 265.845]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    4*obs[11]
    + 3*obs[15]
    + obs[3]*jnp.sin(obs[11] + obs[15] + obs[4]*(obs[11] + obs[15] + obs[9]) + obs[7] + 3*obs[9] + jnp.sin(jnp.sin(obs[8])))
    + 4*obs[4]*(obs[11] + obs[15] + obs[9])
    + obs[7]
    + 7*obs[9]
    + jnp.sin(obs[8])
    + jnp.sin(obs[11] + obs[15] + obs[4]*(obs[11] + obs[15] + obs[9]) + obs[7] + 3*obs[9] + jnp.sin(jnp.sin(obs[8])))
    + 4*jnp.sin(jnp.sin(obs[8])),

    0,

    obs[11]
    + 2*obs[3]*jnp.sin(obs[11] + obs[15] + obs[4]*(obs[11] + obs[15] + obs[9]) + obs[7] + 3*obs[9] + jnp.sin(jnp.sin(obs[8])))
    + obs[5],

    0,
    0,
    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[215.596 226.494 224.309 230.642 214.656 216.992 230.202 113.078 230.675 169.496]


In [ ]:
mujoco_env = MujocoGymnaxWrapper('CheetahRun')
policy = lambda obs: jnp.array([
    obs[1]*obs[14]
    + 2*obs[1]
    + obs[10]*(obs[4] + obs[5])
    + 2*obs[11]
    + 2*obs[12]
    + 2*obs[14]
    + 2*obs[15]
    + 1.61*obs[5]*(obs[1]*obs[14] + 2*obs[1] + obs[10]*(obs[4] + obs[5]) + 2*obs[11] + obs[12] + obs[14] + obs[15])
    + 2.61*obs[9],

    0,

    obs[10]*(obs[4] + obs[5]) + obs[11],

    0,
    0,
    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, mujoco_env)
print(reward_list_inv_pen)

[226.51  220.368 222.571 236.734 192.973 229.68  127.878 165.906 224.036 221.508]
